# Tune `xgb_mspc`

MSPC + XGBoost. Repeated stratified CV GridSearchCV on the train split; writes [`data/processed/tuned/xgb_mspc.json`](../data/processed/tuned/xgb_mspc.json).

**Hyperparameters:** PLS `n_components`, classifier `max_depth`, `learning_rate`.

**Selection:** maximize mean ROC AUC across CV folds.

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd

# Repo root when kernel cwd is SECOM/ or SECOM/tuning/
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tuned_params_path,
)

MODEL_ID = "xgb_mspc"
spec = MODEL_SPECS[MODEL_ID]


In [ ]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


In [ ]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


In [ ]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


In [ ]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
payload = save_tuned_params(spec, cv_summary, fold_results, aggregated)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
print("Best config (mean ROC AUC):")
display(aggregated.head(10))
payload["grid_search_best_params"]
